In [1]:
import pandas as pd
import os
import re
import numpy as np

In [2]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output")

In [3]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [4]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_tornado.csv"))
emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [5]:
print(emissions_df.primary_id.nunique())

49


In [6]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM Value - 

In [7]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

50
48


In [8]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
66782,6067.0,124124.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001754,2046,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001754,NaN
66783,6067.0,124124.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001815,2047,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001815,NaN
66784,6067.0,124124.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001876,2048,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001876,NaN
66785,6067.0,124124.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001935,2049,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001935,NaN
66786,6067.0,124124.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001994,2050,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001994,NaN


In [9]:
# Load decomposed data from original run containing base, wem and wam
original_decomposed_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_original.csv"))
original_decomposed_df.strategy.unique()

array(['Strategy TX:BASE', 'WEM', 'WAM', 'Historical'], dtype=object)

In [10]:
# Keep only base strategy in the original df
original_base_df = original_decomposed_df.loc[original_decomposed_df['strategy'] == 'Strategy TX:BASE']
original_base_df.strategy.unique()

array(['Strategy TX:BASE'], dtype=object)

In [11]:
original_base_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [12]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = pd.concat([original_base_df, filtered_emissions_df], ignore_index=True)
tornado_emissions_df['strategy'].unique()

array(['Strategy TX:BASE',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM Value - 

In [13]:
tornado_emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [14]:
# Keep only relevant CSC.Subsectors

relevant_subsectors = [
    "AG - Crops",
    "AG - Livestock",
    "IN - Industrial Processes",
    "LULUCF - Forest Land",
    "LULUCF - HWP",
    "LULUCF - Wetlands",
    "LULUCF - Cropland",
    "LULUCF - Grassland",
    "LULUCF - Settlements",
    "LULUCF - Other Land",
    "Waste - Solid Waste",
    "Waste - Wastewater Treatment"
]
print(tornado_emissions_df.shape)
tornado_emissions_df = tornado_emissions_df.loc[tornado_emissions_df['CSC.Subsector'].isin(relevant_subsectors)]
print(tornado_emissions_df.shape)

(66787, 16)
(36946, 16)


In [15]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,0.0,0.0,Strategy TX:BASE,135.461601
1,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,105.822139
2,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,135.461601
3,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,135.461601
4,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,134.526574


In [16]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
44,6053.0,120120.0,Singleton - WAM Value - TRNS: Fuel switch medi...,135.461601
45,6054.0,121121.0,Singleton - WAM Value - PFLO: WASO actions to...,130.307543
46,6055.0,122122.0,Singleton - WAM Value - PFLO: WALI actions to...,132.818332
47,6066.0,123123.0,Singleton - WAM Value - WALI: Improved rural w...,134.941811
48,6067.0,124124.0,Singleton - WAM Value - WALI: Improved urban w...,139.440868


In [17]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

49
49


In [18]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 0, 'emission_total'].values[0]
tornado_emissions_agg_df['base_emission_total'] = base_emission_total

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,0.0,0.0,Strategy TX:BASE,135.461601,135.461601,0.000000
1,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,105.822139,135.461601,-29.639462
2,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,135.461601,135.461601,0.000000
3,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,135.461601,135.461601,0.000000
4,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,134.526574,135.461601,-0.935027


In [19]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,0.0,0.0,Strategy TX:BASE,135.461601,135.461601,0.000000,BASE,BASE
1,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,105.822139,135.461601,-29.639462,PFLO,Industrial carbon capture and sequestration
2,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,135.461601,135.461601,0.000000,TRNS,Mode shift passenger vehicles to others
3,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,135.461601,135.461601,0.000000,ENTC,95% of electricity is generated by renewables ...
4,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,134.526574,135.461601,-0.935027,WASO,Increase landfilling


In [20]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [21]:
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-22T16;45;26.395826_tornado_raw.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0


In [22]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [23]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0


In [24]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,1000,AGRC:DEC_CH4_RICE
2,1001,AGRC:DEC_EXPORTS
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE


In [25]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6006
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6006
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6006
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6006
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6006


In [26]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [27]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv', 'pflo'],
      dtype=object)

In [28]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6006
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6006
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6006
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6006
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6006


In [29]:
# filter sectors
target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6006
1,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6006
2,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145955,752911.145955,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6006
3,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257886,746564.257886,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6006
4,PFLO:INC_IND_CCS_WAM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204924,740262.204924,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6006


In [30]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head()

,strategy_id,cb_type,cumulative
0,6006,air_pollution,0.0
1,6006,crop_value,0.0
2,6006,ecosystem_services,0.0
3,6006,env_pollution,0.0
4,6006,fuel_cost,0.0


In [31]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,technical_cost,technical_savings,water_pollution
0,6006,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,-0.571978,NaN,0.0
1,6008,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0
2,6009,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0
3,6010,0.0,NaN,0.0,0.0,3.182296,0.0,0.0,0.000000,0.0,0.0,-0.022323,NaN,0.0
4,6012,0.0,NaN,0.0,0.0,0.351791,0.0,0.0,0.111631,0.0,0.0,-0.066341,NaN,0.0


In [32]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6006,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,-0.571978,NaN,0.0,-0.571978,0.000000,-0.571978
1,6008,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
2,6009,0.0,NaN,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
3,6010,0.0,NaN,0.0,0.0,3.182296,0.0,0.0,0.000000,0.0,0.0,-0.022323,NaN,0.0,3.159973,3.182296,-0.022323
4,6012,0.0,NaN,0.0,0.0,0.351791,0.0,0.0,0.111631,0.0,0.0,-0.066341,NaN,0.0,0.397080,0.463422,-0.066341


## Merge emissions and cb data and save

In [33]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(48, 17)
(49, 8)


In [34]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,consumer_savings,...,human_health,ippu_value,land_pollution,lvst_value,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,105.822139,135.461601,-29.639462,PFLO,Industrial carbon capture and sequestration,0.0,NaN,...,0.0,0.000000,0.0,0.0,-0.571978,NaN,0.0,-0.571978,0.000000,-0.571978
1,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,135.461601,135.461601,0.000000,TRNS,Mode shift passenger vehicles to others,0.0,NaN,...,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
2,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,135.461601,135.461601,0.000000,ENTC,95% of electricity is generated by renewables ...,0.0,NaN,...,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
3,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,134.526574,135.461601,-0.935027,WASO,Increase landfilling,0.0,NaN,...,0.0,0.000000,0.0,0.0,-0.022323,NaN,0.0,3.159973,3.182296,-0.022323
4,6012.0,81081.0,Singleton - WAM Value - WASO: Increase recycling,131.697655,135.461601,-3.763946,WASO,Increase recycling,0.0,NaN,...,0.0,0.111631,0.0,0.0,-0.066341,NaN,0.0,0.397080,0.463422,-0.066341


In [35]:
print(df_merged.shape)

(48, 24)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [36]:
# Update tranformation name for strategy id 6049
df_merged.loc[df_merged['strategy_id'] == 6049, 'transformation_name'] = "Increase solid waste biogas capture"
df_merged.loc[df_merged['strategy_id'] == 6021, 'transformation_name'] = "Shift mode for freight transport"
df_merged.loc[df_merged['strategy_id'] == 6028, 'transformation_name'] = "Shift mode for regional transport"
df_merged.loc[df_merged['strategy_id'] == 6038, 'transformation_name'] = "Increase flaring"

In [37]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot.csv"), index=False)

### Create a QA version

In [38]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,consumer_savings,...,human_health,ippu_value,land_pollution,lvst_value,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,105.822139,135.461601,-29.639462,PFLO,Industrial carbon capture and sequestration,0.0,NaN,...,0.0,0.000000,0.0,0.0,-0.571978,NaN,0.0,-0.571978,0.000000,-0.571978
1,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,135.461601,135.461601,0.000000,TRNS,Mode shift passenger vehicles to others,0.0,NaN,...,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
2,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,135.461601,135.461601,0.000000,ENTC,95% of electricity is generated by renewables ...,0.0,NaN,...,0.0,0.000000,0.0,0.0,0.000000,NaN,0.0,0.000000,0.000000,0.000000
3,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,134.526574,135.461601,-0.935027,WASO,Increase landfilling,0.0,NaN,...,0.0,0.000000,0.0,0.0,-0.022323,NaN,0.0,3.159973,3.182296,-0.022323
4,6012.0,81081.0,Singleton - WAM Value - WASO: Increase recycling,131.697655,135.461601,-3.763946,WASO,Increase recycling,0.0,NaN,...,0.0,0.111631,0.0,0.0,-0.066341,NaN,0.0,0.397080,0.463422,-0.066341


In [39]:
df_merged.sector.unique()

array(['PFLO', 'TRNS', 'ENTC', 'WASO', 'LSMM', 'AGRC', 'FGTV', 'SCOE',
       'CCSQ', 'IPPU', 'INEN', 'SOIL', 'TRDE', 'WALI', 'LVST', 'LNDU'],
      dtype=object)

In [40]:
relevant_sectors = [
    "AGRC",
    "LVST",
    "IPPU",
    "SOIL",
    "WALI",
    "TRWW",
    "WASO",
    "LNDU",
    "LSMM",
    "PFLO"
]

# keep only relevant sectors
df_merged_filtered = df_merged.loc[df_merged['sector'].isin(relevant_sectors)]

relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
]

# keep only relevant fields
df_merged_filtered = df_merged_filtered[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost
0,Industrial carbon capture and sequestration,PFLO,135.461601,105.822139,-29.639462,-0.571978
3,Increase landfilling,WASO,135.461601,134.526574,-0.935027,-0.022323
4,Increase recycling,WASO,135.461601,131.697655,-3.763946,-0.066341
5,Improve manure management for poultry,LSMM,135.461601,134.755329,-0.706272,0.023736
6,Improve rice management,AGRC,135.461601,135.065800,-0.395801,-0.005195


In [41]:
# multiply technical_cost by -1 to get positive costs
df_merged_filtered['technical_cost'] = df_merged_filtered['technical_cost'] * -1

# create marginal total abatement cost column
df_merged_filtered['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged_filtered['technical_cost'] / df_merged_filtered['emission_diff'])*1000*-1

# rename techincal_cost
df_merged_filtered = df_merged_filtered.rename(columns={'technical_cost': 'total_technical_cost_(billion_USD)'})

df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,total_technical_cost_(billion_USD),marginal_total_abatement_cost_(USD/tCO2e)
0,Industrial carbon capture and sequestration,PFLO,135.461601,105.822139,-29.639462,0.571978,19.297841
3,Increase landfilling,WASO,135.461601,134.526574,-0.935027,0.022323,23.874290
4,Increase recycling,WASO,135.461601,131.697655,-3.763946,0.066341,17.625500
5,Improve manure management for poultry,LSMM,135.461601,134.755329,-0.706272,-0.023736,-33.607004
6,Improve rice management,AGRC,135.461601,135.065800,-0.395801,0.005195,13.125749


In [42]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,total_technical_cost_(billion_USD),marginal_total_abatement_cost_(USD/tCO2e)
0,Industrial carbon capture and sequestration,PFLO,135.461601,105.822139,-29.639462,0.571978,19.297841
3,Increase landfilling,WASO,135.461601,134.526574,-0.935027,0.022323,23.874290
4,Increase recycling,WASO,135.461601,131.697655,-3.763946,0.066341,17.625500
5,Improve manure management for poultry,LSMM,135.461601,134.755329,-0.706272,-0.023736,-33.607004
6,Improve rice management,AGRC,135.461601,135.065800,-0.395801,0.005195,13.125749
9,Consumer food waste reduction,WASO,135.461601,134.268024,-1.193577,-0.224826,-188.363526
14,Increase biogas capture at anaerobic decomposi...,LSMM,135.461601,135.445778,-0.015823,0.000316,20.000000
15,Incineration for energy production,WASO,135.461601,135.323416,-0.138185,0.008935,64.661915
17,Reduce use of HFCs,IPPU,135.461601,131.151387,-4.310214,0.064653,15.000000
18,Reduce supply chain losses,AGRC,135.461601,134.166309,-1.295292,-0.257594,-198.869704


In [43]:
df_merged_filtered.to_clipboard(index=False)

In [44]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA.csv"), index=False)